## Dependencias

In [1]:
import joblib
import pandas as pd 
import numpy as np

## Parámetros

In [2]:
SEED = 42

## Lectura conjunto de datos a Predecir

In [3]:
df_a_pred=pd.read_excel(r"C:\Users\Morena\TP2\data\raw\smoking_prediction_entrega.xlsx") # cargo el dataset sin etiquetas

In [4]:
df_a_pred

,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,3.041667,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,4.208333,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,2.041667,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,3.708333,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,N
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,6.625000,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,F,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,...,4.916667,0.502083,0.041667,0.004167,0.583333,0.291667,0.416667,Y,1,Y
5688,55681,F,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,...,3.291667,0.583333,0.041667,0.006250,0.833333,0.500000,0.583333,Y,0,Y
5689,55683,F,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,...,2.625000,0.502778,0.041667,0.003472,0.708333,0.458333,0.500000,Y,0,N
5690,55684,M,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,...,6.083333,0.586111,0.041667,0.004861,0.833333,0.791667,0.750000,Y,0,N


In [5]:
serie_ID = df_a_pred["ID"] #guardo la serie con los datos de la columna "ID", para luego de realizar la predicción, poder rearmar el archivo 
# excel incluyendo el ID.

In [6]:
serie_ID

0       27358
1       27364
2       27368
3       27378
4       27381
        ...  
5687    55676
5688    55681
5689    55683
5690    55684
5691    55691
Name: ID, Length: 5692, dtype: int64

In [7]:
serie_oral = df_a_pred["oral"] #guardo la serie con los datos de la columna "oral", para luego de realizar la predicción, 
# poder rearmar el archivo excel incluyendo esta columna.

In [8]:
serie_oral

0       Y
1       Y
2       Y
3       Y
4       Y
       ..
5687    Y
5688    Y
5689    Y
5690    Y
5691    Y
Name: oral, Length: 5692, dtype: str

## Procesamiento

In [9]:
# Elimino las columnas ID y oral del conjunto de datos ya que durante el entrenamiento de los modelos, no incluí estas 
# variables. Obtengo un DF reducido.

df_a_pred_red = df_a_pred.drop(columns= ["ID", "oral"]) 

In [10]:
#Verifico que se hayan eliminado.

df_a_pred_red 

,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,tartar
0,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,1.958333,3.041667,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,0,Y
1,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,1.500000,4.208333,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,0,N
2,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,2.125000,2.041667,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,1,Y
3,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,1.708333,3.708333,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,0,N
4,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,2.375000,6.625000,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,1,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,F,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,3.125000,4.916667,0.502083,0.041667,0.004167,0.583333,0.291667,0.416667,1,Y
5688,F,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,3.041667,3.291667,0.583333,0.041667,0.006250,0.833333,0.500000,0.583333,0,Y
5689,F,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,3.291667,2.625000,0.502778,0.041667,0.003472,0.708333,0.458333,0.500000,0,N
5690,M,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,2.000000,6.083333,0.586111,0.041667,0.004861,0.833333,0.791667,0.750000,0,N


In [11]:
# listo los nombres de las columnas del conjunto de datos.

df_a_pred_red.columns 

Index(['gender', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)',
       'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)',
       'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol',
       'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein',
       'serum creatinine', 'AST', 'ALT', 'Gtp', 'dental caries', 'tartar'],
      dtype='str')

In [12]:
# defino la variable "cat_cols" con el nombre de las columnas que contienen las variables categóricas que utilice 
# en el procesamiento de datos para entrenar los modelos.

cat_cols = ['gender', 'tartar'] 

In [13]:
cat_cols

['gender', 'tartar']

In [14]:
# cargo el encoder utilizado durante el procesamiento de datos, con el cual convertí las variables categóricas del 
# conjunto de datos de entrenamiento.

ohe = joblib.load(r"C:\Users\Morena\TP2\objects\encoders\ohe.joblib") 

In [15]:
# realizo la misma codificación de datos realizada durante el procesamiento de datos de entrenamiento.

data_codif_ohe = ohe.transform(df_a_pred_red[cat_cols]) 

In [16]:
data_codif_ohe

array([[0., 1., 0., 1.],
       [0., 1., 1., 0.],
       [0., 1., 0., 1.],
       ...,
       [1., 0., 1., 0.],
       [0., 1., 1., 0.],
       [0., 1., 0., 1.]], shape=(5692, 4))

In [17]:
# obtengo los nombres de las columnas generadas por el encoder.

ohe_columns = ohe.get_feature_names_out(cat_cols) 

In [18]:
ohe_columns

array(['gender_F', 'gender_M', 'tartar_N', 'tartar_Y'], dtype=object)

In [19]:
# genero un DataFrame con los datos convertidos y el nombre de sus columnas, respetando el índice del DF con los datos
# a predecir.

data_codif_ohe_df = pd.DataFrame(data_codif_ohe, columns=ohe_columns, index=df_a_pred.index) 

In [20]:
# verifico la estructura del DF generado.

data_codif_ohe_df 

,gender_F,gender_M,tartar_N,tartar_Y
0,0.0,1.0,0.0,1.0
1,0.0,1.0,1.0,0.0
2,0.0,1.0,0.0,1.0
3,0.0,1.0,1.0,0.0
4,0.0,1.0,0.0,1.0
...,...,...,...,...
5687,1.0,0.0,0.0,1.0
5688,1.0,0.0,0.0,1.0
5689,1.0,0.0,1.0,0.0
5690,0.0,1.0,1.0,0.0


In [21]:
# al DataFrame reducido (sin las columnas "ID" y "oral"), le elimino las columnas categóricas almacenadas en "cat_cols".

df_a_pred_red_num = df_a_pred_red.drop(columns=cat_cols)

In [22]:
# verifico que todas las variables que quedaron, sean numéricas.

df_a_pred_red_num.info()

<class 'pandas.DataFrame'>
RangeIndex: 5692 entries, 0 to 5691
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  5692 non-null   int64  
 1   height(cm)           5692 non-null   int64  
 2   weight(kg)           5692 non-null   int64  
 3   waist(cm)            5692 non-null   float64
 4   eyesight(left)       5692 non-null   float64
 5   eyesight(right)      5692 non-null   float64
 6   hearing(left)        5692 non-null   float64
 7   hearing(right)       5692 non-null   float64
 8   systolic             5692 non-null   float64
 9   relaxation           5692 non-null   float64
 10  fasting blood sugar  5692 non-null   float64
 11  Cholesterol          5692 non-null   float64
 12  triglyceride         5692 non-null   float64
 13  HDL                  5692 non-null   float64
 14  LDL                  5692 non-null   float64
 15  hemoglobin           5692 non-null   float64
 16 

In [23]:
# verifico la información del DF generado con OHE.

data_codif_ohe_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5692 entries, 0 to 5691
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   gender_F  5692 non-null   float64
 1   gender_M  5692 non-null   float64
 2   tartar_N  5692 non-null   float64
 3   tartar_Y  5692 non-null   float64
dtypes: float64(4)
memory usage: 178.0 KB


In [24]:
# genero un DataFrame con la misma estructura que el utilizado para entrenar los modelos, concatenando el DF a predecir,
# con sus datos ya procesados, y el DF generado con OHE.

df_a_pred_final = pd.concat([df_a_pred_red_num, data_codif_ohe_df], axis=1)

In [25]:
df_a_pred_final

,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,gender_F,gender_M,tartar_N,tartar_Y
0,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,3.125000,...,0.041667,0.006250,0.750000,0.708333,0.708333,0,0.0,1.0,0.0,1.0
1,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,3.250000,...,0.041667,0.041667,0.791667,1.125000,1.333333,0,0.0,1.0,1.0,0.0
2,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,3.458333,...,0.041667,0.006250,1.083333,1.291667,2.000000,1,0.0,1.0,0.0,1.0
3,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,3.500000,...,0.041667,0.043750,0.833333,0.583333,0.458333,0,0.0,1.0,1.0,0.0
4,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,3.000000,...,0.041667,0.041667,1.250000,1.625000,1.958333,1,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,2.833333,...,0.041667,0.004167,0.583333,0.291667,0.416667,1,1.0,0.0,0.0,1.0
5688,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,2.583333,...,0.041667,0.006250,0.833333,0.500000,0.583333,0,1.0,0.0,0.0,1.0
5689,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,3.000000,...,0.041667,0.003472,0.708333,0.458333,0.500000,0,1.0,0.0,1.0,0.0
5690,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,3.166667,...,0.041667,0.004861,0.833333,0.791667,0.750000,0,0.0,1.0,1.0,0.0


## Predicciones

### Arbol de decisión

In [26]:
clf_dtc = joblib.load(r"C:\Users\Morena\TP2\objects\models\modelo_arbol_decision.pkl")

In [27]:
clf_dtc

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at 

In [28]:
y_pred_dtc = clf_dtc.predict(df_a_pred_final)

In [29]:
y_pred_dtc

array([0, 1, 1, ..., 0, 0, 1], shape=(5692,))

### Random Forest

In [30]:
# cargo el modelo entrenado con Random Forest

clf_rf = joblib.load(r"C:\Users\Morena\TP2\objects\models\modelo_random_forest.pkl")

In [31]:
clf_rf

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",137
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",31
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",300
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [32]:
# verifico que los nombres de las columnas almacenadas en el modelo, coincidan con el conjunto de datos que queremos 
# predecir ("df_a_pred_final").

print(clf_rf.feature_names_in_)

['age' 'height(cm)' 'weight(kg)' 'waist(cm)' 'eyesight(left)'
 'eyesight(right)' 'hearing(left)' 'hearing(right)' 'systolic'
 'relaxation' 'fasting blood sugar' 'Cholesterol' 'triglyceride' 'HDL'
 'LDL' 'hemoglobin' 'Urine protein' 'serum creatinine' 'AST' 'ALT' 'Gtp'
 'dental caries' 'gender_F' 'gender_M' 'tartar_N' 'tartar_Y']


In [33]:
# relizo la predicción utilizando el modelo.

y_pred_clf_rf = clf_rf.predict(df_a_pred_final)

In [34]:
# verifico los valores generados por el modelo en la variable "y_pred_clf_rf".

y_pred_clf_rf

array([0, 1, 1, ..., 0, 0, 1], shape=(5692,))

### XGBoost

In [35]:
clf_xgb = joblib.load(r"C:\Users\Morena\TP2\objects\models\modelo_XGBoost.pkl")

In [36]:
clf_xgb

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [37]:
y_pred_clf_xgb = clf_xgb.predict(df_a_pred_final)

In [38]:
y_pred_clf_xgb

array([0, 0, 1, ..., 0, 0, 1], shape=(5692,))

In [39]:
y_pred_clf_xgb_proba = clf_xgb.predict_proba(df_a_pred_final)[:,1]

In [40]:
y_pred_clf_xgb_proba

array([0.40420306, 0.48891786, 0.7616844 , ..., 0.03373881, 0.38737565,
       0.5859438 ], shape=(5692,), dtype=float32)

In [41]:
threshold = 0.435

In [42]:
y_pred_clf_xgb_proba_custom = (y_pred_clf_xgb_proba >= threshold).astype(int)

In [43]:
y_pred_clf_xgb_proba_custom

array([0, 1, 1, ..., 0, 0, 1], shape=(5692,))

## Rearmado del conjunto de datos con columna "smoking_prediction" en el conjunto de Datos a Predecir

### Con Arbol de Decisión

In [44]:
# realizo una copia del DF a predecir, ya procesado, y a esta nueva copia, la llamo "df_a_pred_final_rf".

df_a_pred_final_dtc = df_a_pred_final.copy()

In [45]:
# chequeo que coincidan los datos con el DF copiado.

df_a_pred_final_dtc

,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,gender_F,gender_M,tartar_N,tartar_Y
0,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,3.125000,...,0.041667,0.006250,0.750000,0.708333,0.708333,0,0.0,1.0,0.0,1.0
1,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,3.250000,...,0.041667,0.041667,0.791667,1.125000,1.333333,0,0.0,1.0,1.0,0.0
2,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,3.458333,...,0.041667,0.006250,1.083333,1.291667,2.000000,1,0.0,1.0,0.0,1.0
3,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,3.500000,...,0.041667,0.043750,0.833333,0.583333,0.458333,0,0.0,1.0,1.0,0.0
4,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,3.000000,...,0.041667,0.041667,1.250000,1.625000,1.958333,1,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,2.833333,...,0.041667,0.004167,0.583333,0.291667,0.416667,1,1.0,0.0,0.0,1.0
5688,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,2.583333,...,0.041667,0.006250,0.833333,0.500000,0.583333,0,1.0,0.0,0.0,1.0
5689,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,3.000000,...,0.041667,0.003472,0.708333,0.458333,0.500000,0,1.0,0.0,1.0,0.0
5690,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,3.166667,...,0.041667,0.004861,0.833333,0.791667,0.750000,0,0.0,1.0,1.0,0.0


In [46]:
# le agrego al DataFrame la columna con los datos devueltos por la predicción del modelo Arbol de Decisión.

df_a_pred_final_dtc["smoking_prediction"] = y_pred_dtc

In [47]:
# chequeo que se haya agregado.

df_a_pred_final_dtc

,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,serum creatinine,AST,ALT,Gtp,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction
0,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,3.125000,...,0.006250,0.750000,0.708333,0.708333,0,0.0,1.0,0.0,1.0,0
1,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,3.250000,...,0.041667,0.791667,1.125000,1.333333,0,0.0,1.0,1.0,0.0,1
2,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,3.458333,...,0.006250,1.083333,1.291667,2.000000,1,0.0,1.0,0.0,1.0,1
3,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,3.500000,...,0.043750,0.833333,0.583333,0.458333,0,0.0,1.0,1.0,0.0,0
4,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,3.000000,...,0.041667,1.250000,1.625000,1.958333,1,0.0,1.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,2.833333,...,0.004167,0.583333,0.291667,0.416667,1,1.0,0.0,0.0,1.0,0
5688,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,2.583333,...,0.006250,0.833333,0.500000,0.583333,0,1.0,0.0,0.0,1.0,0
5689,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,3.000000,...,0.003472,0.708333,0.458333,0.500000,0,1.0,0.0,1.0,0.0,0
5690,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,3.166667,...,0.004861,0.833333,0.791667,0.750000,0,0.0,1.0,1.0,0.0,0


In [48]:
# Le inserto al DF, la columna "ID" en la posicion "0"

df_a_pred_final_dtc.insert(loc=0, column="ID", value = serie_ID)

In [49]:
# Le inserto al DF, la columna "oral" en la posicion "22"

df_a_pred_final_dtc.insert(loc=22, column="oral", value = serie_oral)

In [50]:
# verifico como quedó la estructura.

df_a_pred_final_dtc

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,AST,ALT,Gtp,oral,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction
0,27358,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.750000,0.708333,0.708333,Y,0,0.0,1.0,0.0,1.0,0
1,27364,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,0.791667,1.125000,1.333333,Y,0,0.0,1.0,1.0,0.0,1
2,27368,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,1.083333,1.291667,2.000000,Y,1,0.0,1.0,0.0,1.0,1
3,27378,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.833333,0.583333,0.458333,Y,0,0.0,1.0,1.0,0.0,0
4,27381,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,1.250000,1.625000,1.958333,Y,1,0.0,1.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,0.583333,0.291667,0.416667,Y,1,1.0,0.0,0.0,1.0,0
5688,55681,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,0.833333,0.500000,0.583333,Y,0,1.0,0.0,0.0,1.0,0
5689,55683,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,0.708333,0.458333,0.500000,Y,0,1.0,0.0,1.0,0.0,0
5690,55684,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,0.833333,0.791667,0.750000,Y,0,0.0,1.0,1.0,0.0,0


In [51]:
# revierto la transformacion realizada con OHE.

reversion_ohe = ohe.inverse_transform(data_codif_ohe_df)

In [52]:
# verifico los datos

reversion_ohe

array([['M', 'Y'],
       ['M', 'N'],
       ['M', 'Y'],
       ...,
       ['F', 'N'],
       ['M', 'N'],
       ['M', 'Y']], shape=(5692, 2), dtype=object)

In [53]:
# lo convierto a un DF agregando las columnas de las variables categóricas que se habian transformado.

reversion_ohe_df = pd.DataFrame(reversion_ohe, columns=cat_cols, index=df_a_pred.index) 

In [54]:
# Verifico el DF.

reversion_ohe_df

,gender,tartar
0,M,Y
1,M,N
2,M,Y
3,M,N
4,M,Y
...,...,...
5687,F,Y
5688,F,Y
5689,F,N
5690,M,N


In [55]:
# concateno ambos DF.

df_pred_dtc = pd.concat([df_a_pred_final_dtc, reversion_ohe_df], axis=1)

In [56]:
# verifico

df_pred_dtc

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,Gtp,oral,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction,gender,tartar
0,27358,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.708333,Y,0,0.0,1.0,0.0,1.0,0,M,Y
1,27364,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,1.333333,Y,0,0.0,1.0,1.0,0.0,1,M,N
2,27368,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,2.000000,Y,1,0.0,1.0,0.0,1.0,1,M,Y
3,27378,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.458333,Y,0,0.0,1.0,1.0,0.0,0,M,N
4,27381,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,1.958333,Y,1,0.0,1.0,0.0,1.0,1,M,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,0.416667,Y,1,1.0,0.0,0.0,1.0,0,F,Y
5688,55681,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,0.583333,Y,0,1.0,0.0,0.0,1.0,0,F,Y
5689,55683,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,0.500000,Y,0,1.0,0.0,1.0,0.0,0,F,N
5690,55684,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,0.750000,Y,0,0.0,1.0,1.0,0.0,0,M,N


In [57]:
# elimino las columas que había agregado con OHE.

df_pred_dtc = df_pred_dtc.drop(columns=ohe_columns)

In [58]:
df_pred_dtc

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,smoking_prediction,gender,tartar
0,27358,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,0,M,Y
1,27364,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,1,M,N
2,27368,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,1,M,Y
3,27378,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,0,M,N
4,27381,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,1,M,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,0.041667,0.004167,0.583333,0.291667,0.416667,Y,1,0,F,Y
5688,55681,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,0.041667,0.006250,0.833333,0.500000,0.583333,Y,0,0,F,Y
5689,55683,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,0.041667,0.003472,0.708333,0.458333,0.500000,Y,0,0,F,N
5690,55684,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,0.041667,0.004861,0.833333,0.791667,0.750000,Y,0,0,M,N


In [59]:
# quito la columna "gender" del DF, convirtiendola en una serie pandas

col_gender = df_pred_dtc.pop("gender")

In [60]:
# para luego insertarla en la posición 1 (como estaba originalmente en el conjunto de datos a predecir).

df_pred_dtc.insert(loc=1, column="gender", value = col_gender)

In [61]:
# hago lo mismo con la columna "tartar".

col_tartar = df_pred_dtc.pop("tartar")

In [62]:
# reinsertandola en la posición 25.

df_pred_dtc.insert(loc=25, column="tartar", value = col_tartar)

In [63]:
df_pred_dtc

,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking_prediction
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y,0
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N,1
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y,1
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,N,0
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,Y,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,F,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,...,0.502083,0.041667,0.004167,0.583333,0.291667,0.416667,Y,1,Y,0
5688,55681,F,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,...,0.583333,0.041667,0.006250,0.833333,0.500000,0.583333,Y,0,Y,0
5689,55683,F,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,...,0.502778,0.041667,0.003472,0.708333,0.458333,0.500000,Y,0,N,0
5690,55684,M,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,...,0.586111,0.041667,0.004861,0.833333,0.791667,0.750000,Y,0,N,0


In [64]:
# ahora que ya tendo el Data Frame con la misma estructura que el conjunto de datos a predecir, con el agregado de la 
# columna "smoking_prediction", genero el archivo excel final, sin incluir el indice del DF.
# lo guardo en la carpeta TP2\data\predictions" con el nombre "smoking_prediction_entrega_dtc.xlsx"

In [65]:
df_pred_dtc.to_excel(r"C:\Users\Morena\TP2\data\predictions\smoking_prediction_entrega_dtc.xlsx", index=False)

### Con Random Forest

In [66]:
# realizo una copia del DF a predecir, ya procesado, y a esta nueva copia, la llamo "df_a_pred_final_rf".

df_a_pred_final_rf = df_a_pred_final.copy()

In [67]:
# chequeo que coincidan los datos con el DF copiado.

df_a_pred_final_rf

,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,gender_F,gender_M,tartar_N,tartar_Y
0,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,3.125000,...,0.041667,0.006250,0.750000,0.708333,0.708333,0,0.0,1.0,0.0,1.0
1,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,3.250000,...,0.041667,0.041667,0.791667,1.125000,1.333333,0,0.0,1.0,1.0,0.0
2,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,3.458333,...,0.041667,0.006250,1.083333,1.291667,2.000000,1,0.0,1.0,0.0,1.0
3,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,3.500000,...,0.041667,0.043750,0.833333,0.583333,0.458333,0,0.0,1.0,1.0,0.0
4,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,3.000000,...,0.041667,0.041667,1.250000,1.625000,1.958333,1,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,2.833333,...,0.041667,0.004167,0.583333,0.291667,0.416667,1,1.0,0.0,0.0,1.0
5688,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,2.583333,...,0.041667,0.006250,0.833333,0.500000,0.583333,0,1.0,0.0,0.0,1.0
5689,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,3.000000,...,0.041667,0.003472,0.708333,0.458333,0.500000,0,1.0,0.0,1.0,0.0
5690,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,3.166667,...,0.041667,0.004861,0.833333,0.791667,0.750000,0,0.0,1.0,1.0,0.0


In [68]:
# le agrego al DataFrame la columna con los datos devueltos por la predicción del modelo Random Forest.

df_a_pred_final_rf["smoking_prediction"] = y_pred_clf_rf

In [69]:
# chequeo que se haya agregado.

df_a_pred_final_rf

,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,serum creatinine,AST,ALT,Gtp,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction
0,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,3.125000,...,0.006250,0.750000,0.708333,0.708333,0,0.0,1.0,0.0,1.0,0
1,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,3.250000,...,0.041667,0.791667,1.125000,1.333333,0,0.0,1.0,1.0,0.0,1
2,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,3.458333,...,0.006250,1.083333,1.291667,2.000000,1,0.0,1.0,0.0,1.0,1
3,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,3.500000,...,0.043750,0.833333,0.583333,0.458333,0,0.0,1.0,1.0,0.0,0
4,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,3.000000,...,0.041667,1.250000,1.625000,1.958333,1,0.0,1.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,2.833333,...,0.004167,0.583333,0.291667,0.416667,1,1.0,0.0,0.0,1.0,0
5688,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,2.583333,...,0.006250,0.833333,0.500000,0.583333,0,1.0,0.0,0.0,1.0,0
5689,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,3.000000,...,0.003472,0.708333,0.458333,0.500000,0,1.0,0.0,1.0,0.0,0
5690,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,3.166667,...,0.004861,0.833333,0.791667,0.750000,0,0.0,1.0,1.0,0.0,0


In [70]:
# Le inserto al DF, la columna "ID" en la posicion "0"

df_a_pred_final_rf.insert(loc=0, column="ID", value = serie_ID)

In [71]:
# Le inserto al DF, la columna "oral" en la posicion "22"

df_a_pred_final_rf.insert(loc=22, column="oral", value = serie_oral)

In [72]:
# verifico como quedó la estructura.

df_a_pred_final_rf

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,AST,ALT,Gtp,oral,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction
0,27358,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.750000,0.708333,0.708333,Y,0,0.0,1.0,0.0,1.0,0
1,27364,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,0.791667,1.125000,1.333333,Y,0,0.0,1.0,1.0,0.0,1
2,27368,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,1.083333,1.291667,2.000000,Y,1,0.0,1.0,0.0,1.0,1
3,27378,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.833333,0.583333,0.458333,Y,0,0.0,1.0,1.0,0.0,0
4,27381,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,1.250000,1.625000,1.958333,Y,1,0.0,1.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,0.583333,0.291667,0.416667,Y,1,1.0,0.0,0.0,1.0,0
5688,55681,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,0.833333,0.500000,0.583333,Y,0,1.0,0.0,0.0,1.0,0
5689,55683,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,0.708333,0.458333,0.500000,Y,0,1.0,0.0,1.0,0.0,0
5690,55684,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,0.833333,0.791667,0.750000,Y,0,0.0,1.0,1.0,0.0,0


In [73]:
# revierto la transformacion realizada con OHE.

reversion_ohe = ohe.inverse_transform(data_codif_ohe_df)

In [74]:
# verifico los datos

reversion_ohe

array([['M', 'Y'],
       ['M', 'N'],
       ['M', 'Y'],
       ...,
       ['F', 'N'],
       ['M', 'N'],
       ['M', 'Y']], shape=(5692, 2), dtype=object)

In [75]:
# lo convierto a un DF agregando las columnas de las variables categóricas que se habian transformado.

reversion_ohe_df = pd.DataFrame(reversion_ohe, columns=cat_cols, index=df_a_pred.index) 

In [76]:
# Verifico el DF.

reversion_ohe_df

,gender,tartar
0,M,Y
1,M,N
2,M,Y
3,M,N
4,M,Y
...,...,...
5687,F,Y
5688,F,Y
5689,F,N
5690,M,N


In [77]:
# concateno ambos DF.

df_pred_rf = pd.concat([df_a_pred_final_rf, reversion_ohe_df], axis=1)

In [78]:
# verifico

df_pred_rf

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,Gtp,oral,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction,gender,tartar
0,27358,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.708333,Y,0,0.0,1.0,0.0,1.0,0,M,Y
1,27364,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,1.333333,Y,0,0.0,1.0,1.0,0.0,1,M,N
2,27368,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,2.000000,Y,1,0.0,1.0,0.0,1.0,1,M,Y
3,27378,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.458333,Y,0,0.0,1.0,1.0,0.0,0,M,N
4,27381,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,1.958333,Y,1,0.0,1.0,0.0,1.0,1,M,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,0.416667,Y,1,1.0,0.0,0.0,1.0,0,F,Y
5688,55681,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,0.583333,Y,0,1.0,0.0,0.0,1.0,0,F,Y
5689,55683,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,0.500000,Y,0,1.0,0.0,1.0,0.0,0,F,N
5690,55684,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,0.750000,Y,0,0.0,1.0,1.0,0.0,0,M,N


In [79]:
# elimino las columas que había agregado con OHE.

df_pred_rf = df_pred_rf.drop(columns=ohe_columns)

In [80]:
df_pred_rf

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,smoking_prediction,gender,tartar
0,27358,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,0,M,Y
1,27364,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,1,M,N
2,27368,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,1,M,Y
3,27378,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,0,M,N
4,27381,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,1,M,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,0.041667,0.004167,0.583333,0.291667,0.416667,Y,1,0,F,Y
5688,55681,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,0.041667,0.006250,0.833333,0.500000,0.583333,Y,0,0,F,Y
5689,55683,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,0.041667,0.003472,0.708333,0.458333,0.500000,Y,0,0,F,N
5690,55684,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,0.041667,0.004861,0.833333,0.791667,0.750000,Y,0,0,M,N


In [81]:
# quito la columna "gender" del DF, convirtiendola en una serie pandas

col_gender = df_pred_rf.pop("gender")

In [82]:
# para luego insertarla en la posición 1 (como estaba originalmente en el conjunto de datos a predecir).

df_pred_rf.insert(loc=1, column="gender", value = col_gender)

In [83]:
# verifico el cambio.

df_pred_rf

,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,smoking_prediction,tartar
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,0,Y
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,1,N
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,1,Y
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,0,N
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,1,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,F,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,...,0.502083,0.041667,0.004167,0.583333,0.291667,0.416667,Y,1,0,Y
5688,55681,F,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,...,0.583333,0.041667,0.006250,0.833333,0.500000,0.583333,Y,0,0,Y
5689,55683,F,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,...,0.502778,0.041667,0.003472,0.708333,0.458333,0.500000,Y,0,0,N
5690,55684,M,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,...,0.586111,0.041667,0.004861,0.833333,0.791667,0.750000,Y,0,0,N


In [84]:
# hago lo mismo con la columna "tartar".

col_tartar = df_pred_rf.pop("tartar")

In [85]:
# reinsertandola en la posición 25.

df_pred_rf.insert(loc=25, column="tartar", value = col_tartar)

In [86]:
df_pred_rf

,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking_prediction
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y,0
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N,1
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y,1
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,N,0
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,Y,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,F,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,...,0.502083,0.041667,0.004167,0.583333,0.291667,0.416667,Y,1,Y,0
5688,55681,F,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,...,0.583333,0.041667,0.006250,0.833333,0.500000,0.583333,Y,0,Y,0
5689,55683,F,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,...,0.502778,0.041667,0.003472,0.708333,0.458333,0.500000,Y,0,N,0
5690,55684,M,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,...,0.586111,0.041667,0.004861,0.833333,0.791667,0.750000,Y,0,N,0


In [87]:
# ahora que ya tendo el Data Frame con la misma estructura que el conjunto de datos a predecir, con el agregado de la 
# columna "smoking_prediction", genero el archivo excel final, sin incluir el indice del DF.
# lo guardo en la carpeta TP2\data\predictions" con el nombre "smoking_prediction_entrega_rf.xlsx"

In [88]:
df_pred_rf.to_excel(r"C:\Users\Morena\TP2\data\predictions\smoking_prediction_entrega_rf.xlsx", index=False)

### Con XGBoost

In [89]:
# realizo una copia del DF a predecir, ya procesado, y a esta nueva copia, la llamo "df_a_pred_final_rf".

df_a_pred_final_xgb = df_a_pred_final.copy()

In [90]:
# le agrego al DataFrame la columna con los datos devueltos por la predicción del modelo Random Forest.

df_a_pred_final_xgb["smoking_prediction"] = y_pred_clf_xgb

In [91]:
df_a_pred_final_xgb

,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,...,serum creatinine,AST,ALT,Gtp,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction
0,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,3.125000,...,0.006250,0.750000,0.708333,0.708333,0,0.0,1.0,0.0,1.0,0
1,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,3.250000,...,0.041667,0.791667,1.125000,1.333333,0,0.0,1.0,1.0,0.0,0
2,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,3.458333,...,0.006250,1.083333,1.291667,2.000000,1,0.0,1.0,0.0,1.0,1
3,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,3.500000,...,0.043750,0.833333,0.583333,0.458333,0,0.0,1.0,1.0,0.0,0
4,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,3.000000,...,0.041667,1.250000,1.625000,1.958333,1,0.0,1.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,2.833333,...,0.004167,0.583333,0.291667,0.416667,1,1.0,0.0,0.0,1.0,0
5688,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,2.583333,...,0.006250,0.833333,0.500000,0.583333,0,1.0,0.0,0.0,1.0,0
5689,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,3.000000,...,0.003472,0.708333,0.458333,0.500000,0,1.0,0.0,1.0,0.0,0
5690,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,3.166667,...,0.004861,0.833333,0.791667,0.750000,0,0.0,1.0,1.0,0.0,0


In [92]:
# Le inserto al DF, la columna "ID" en la posicion "0"

df_a_pred_final_xgb.insert(loc=0, column="ID", value = serie_ID)

In [93]:
# Le inserto al DF, la columna "oral" en la posicion "22"

df_a_pred_final_xgb.insert(loc=22, column="oral", value = serie_oral)

In [94]:
df_a_pred_final_xgb

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,AST,ALT,Gtp,oral,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction
0,27358,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.750000,0.708333,0.708333,Y,0,0.0,1.0,0.0,1.0,0
1,27364,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,0.791667,1.125000,1.333333,Y,0,0.0,1.0,1.0,0.0,0
2,27368,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,1.083333,1.291667,2.000000,Y,1,0.0,1.0,0.0,1.0,1
3,27378,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.833333,0.583333,0.458333,Y,0,0.0,1.0,1.0,0.0,0
4,27381,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,1.250000,1.625000,1.958333,Y,1,0.0,1.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,0.583333,0.291667,0.416667,Y,1,1.0,0.0,0.0,1.0,0
5688,55681,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,0.833333,0.500000,0.583333,Y,0,1.0,0.0,0.0,1.0,0
5689,55683,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,0.708333,0.458333,0.500000,Y,0,1.0,0.0,1.0,0.0,0
5690,55684,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,0.833333,0.791667,0.750000,Y,0,0.0,1.0,1.0,0.0,0


In [95]:
# revierto la transformacion realizada con OHE.

reversion_ohe = ohe.inverse_transform(data_codif_ohe_df)

In [96]:
reversion_ohe

array([['M', 'Y'],
       ['M', 'N'],
       ['M', 'Y'],
       ...,
       ['F', 'N'],
       ['M', 'N'],
       ['M', 'Y']], shape=(5692, 2), dtype=object)

In [97]:
# lo convierto a un DF agregando las columnas de las variables categóricas que se habian transformado.

reversion_ohe_df = pd.DataFrame(reversion_ohe, columns=cat_cols, index=df_a_pred.index) 

In [98]:
reversion_ohe_df

,gender,tartar
0,M,Y
1,M,N
2,M,Y
3,M,N
4,M,Y
...,...,...
5687,F,Y
5688,F,Y
5689,F,N
5690,M,N


In [99]:
# concateno ambos DF.

df_pred_xgb = pd.concat([df_a_pred_final_xgb, reversion_ohe_df], axis=1)

In [100]:
df_pred_xgb

,ID,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,Gtp,oral,dental caries,gender_F,gender_M,tartar_N,tartar_Y,smoking_prediction,gender,tartar
0,27358,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,5.041667,...,0.708333,Y,0,0.0,1.0,0.0,1.0,0,M,Y
1,27364,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,5.375000,...,1.333333,Y,0,0.0,1.0,1.0,0.0,0,M,N
2,27368,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,5.250000,...,2.000000,Y,1,0.0,1.0,0.0,1.0,1,M,Y
3,27378,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,5.333333,...,0.458333,Y,0,0.0,1.0,1.0,0.0,0,M,N
4,27381,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,4.500000,...,1.958333,Y,1,0.0,1.0,0.0,1.0,1,M,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,4.583333,...,0.416667,Y,1,1.0,0.0,0.0,1.0,0,F,Y
5688,55681,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,4.208333,...,0.583333,Y,0,1.0,0.0,0.0,1.0,0,F,Y
5689,55683,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,4.875000,...,0.500000,Y,0,1.0,0.0,1.0,0.0,0,F,N
5690,55684,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,5.541667,...,0.750000,Y,0,0.0,1.0,1.0,0.0,0,M,N


In [101]:
# elimino las columas que había agregado con OHE.

df_pred_xgb = df_pred_xgb.drop(columns=ohe_columns)

In [102]:
# quito la columna "gender" del DF, convirtiendola en una serie pandas

col_gender = df_pred_xgb.pop("gender")

In [103]:
# para luego insertarla en la posición 1 (como estaba originalmente en el conjunto de datos a predecir).

df_pred_xgb.insert(loc=1, column="gender", value = col_gender)

In [104]:
# hago lo mismo con la columna "tartar".

col_tartar = df_pred_xgb.pop("tartar")

In [105]:
# reinsertandola en la posición 25.

df_pred_xgb.insert(loc=25, column="tartar", value = col_tartar)

In [106]:
df_pred_xgb

,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking_prediction
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y,0
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N,0
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y,1
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,N,0
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,Y,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5687,55676,F,40,170,65,3.125000,0.006250,0.006250,0.041667,0.041667,...,0.502083,0.041667,0.004167,0.583333,0.291667,0.416667,Y,1,Y,0
5688,55681,F,45,160,50,2.916667,0.043056,0.043056,0.041667,0.041667,...,0.583333,0.041667,0.006250,0.833333,0.500000,0.583333,Y,0,Y,0
5689,55683,F,55,160,50,2.836806,0.041667,0.043056,0.041667,0.041667,...,0.502778,0.041667,0.003472,0.708333,0.458333,0.500000,Y,0,N,0
5690,55684,M,60,165,60,3.250000,0.005556,0.041667,0.041667,0.041667,...,0.586111,0.041667,0.004861,0.833333,0.791667,0.750000,Y,0,N,0


In [107]:
# ahora que ya tendo el Data Frame con la misma estructura que el conjunto de datos a predecir, con el agregado de la 
# columna "smoking_prediction", genero el archivo excel final, sin incluir el indice del DF.
# lo guardo en la carpeta TP2\data\predictions" con el nombre "smoking_prediction_entrega_xgb.xlsx"

In [108]:
df_pred_xgb.to_excel(r"C:\Users\Morena\TP2\data\predictions\smoking_prediction_entrega_xgb.xlsx", index=False)